In [3]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# Load the API key from your .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Confirm it loaded correctly
if OPENAI_API_KEY:
    print("✅ API key loaded successfully")
else:
    print("❌ API key not found — check your .env file")


✅ API key loaded successfully


In [4]:
pdf_folder = "./pdfs"
all_docs = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        path = os.path.join(pdf_folder, filename)
        loader = PyPDFLoader(path)
        docs = loader.load()
        all_docs.extend(docs)
        print(f"📄 Loaded: {filename} — {len(docs)} pages")

print(f"\n✅ Total pages loaded across all PDFs: {len(all_docs)}")

📄 Loaded: 7-Agentforce-Lessons-from-15-Enterprise-Deployments.pdf — 23 pages
📄 Loaded: IDC AI Market place report.pdf — 14 pages
📄 Loaded: wp-how-to-champion-ai-as-an-executive.pdf — 8 pages

✅ Total pages loaded across all PDFs: 45


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_docs)

# Metrics for your report
total_chunks = len(chunks)
avg_chunk_size = sum(len(c.page_content) for c in chunks) / total_chunks

print(f"✅ Chunking complete")
print(f"   Total chunks created : {total_chunks}")
print(f"   Average chunk size   : {avg_chunk_size:.1f} characters")
print(f"\n── Sample chunk (first one) ──────────────────────────────")
print(chunks[0].page_content)

✅ Chunking complete
   Total chunks created : 188
   Average chunk size   : 413.5 characters

── Sample chunk (first one) ──────────────────────────────
7 
15+ ENTERPRISE 
DEPLOYMENTS
AGENTFORCE 
LESSONS FROM 
Bluprintx 2026


In [6]:
print("⏳ Generating embeddings — please wait...")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore.save_local("faiss_index")

print("✅ Embeddings generated and saved to faiss_index/ folder")

⏳ Generating embeddings — please wait...
✅ Embeddings generated and saved to faiss_index/ folder


In [7]:
from langchain_classic.chains import RetrievalQA

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model="gpt-3.5-turbo", openai_api_key=OPENAI_API_KEY)

qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# ✏️ Change this to a question relevant to YOUR documents
query = "What are the key lessons for enterprise AI deployments?"

result = qa_chain.invoke(query)

print(f"🔍 Query : {query}")
print(f"\n💬 Answer:\n{result['result']}")

🔍 Query : What are the key lessons for enterprise AI deployments?

💬 Answer:
Some key lessons for enterprise AI deployments include:

1. Investing in AI early can deliver quick wins and prepare teams for long-term transformation.
2. Research and experiment with AI solutions to understand costs, features, use cases, and scalability.
3. Engage with vendors to gain insight into different AI tools and solutions.
4. Experiment with pilot projects to uncover potential pitfalls like cultural resistance, workflow disruptions, or skill gaps.
5. Move from pilot projects to production by aligning workflows to deliver measurable outcomes.

These lessons can help enterprises successfully deploy AI solutions and leverage their capabilities effectively.


In [8]:
top_docs = retriever.invoke(query)

print(f"Top {len(top_docs)} chunks retrieved for: '{query}'")
print("=" * 60)

for i, doc in enumerate(top_docs, 1):
    source = doc.metadata.get('source', 'Unknown')
    page = doc.metadata.get('page', '?')
    print(f"\n── Chunk {i} ──────────────────────────────────────────────")
    print(f"Source : {os.path.basename(source)}")
    print(f"Page   : {page}")
    print(f"\nContent preview:")
    print(doc.page_content[:400])
    print("-" * 60)

print("\n✅ Pipeline complete — all cells ran successfully")

Top 3 chunks retrieved for: 'What are the key lessons for enterprise AI deployments?'

── Chunk 1 ──────────────────────────────────────────────
Source : wp-how-to-champion-ai-as-an-executive.pdf
Page   : 4

Content preview:
a significant impact. Investing in AI early serves two 
purposes, one it delivers quick wins while preparing 
teams for long-term, AI-driven transformation. Second, 
it exposes potential pitfalls, such as cultural resistance, 
workflow disruptions, or skill gaps, before larger 
investments are made. For example, deploying an AI 
solution to streamline a repetitive process can reveal 
Page 5© Oracl
------------------------------------------------------------

── Chunk 2 ──────────────────────────────────────────────
Source : wp-how-to-champion-ai-as-an-executive.pdf
Page   : 5

Content preview:
Research and Experiment With AI Solutions  
Explore the AI products available in the market and 
engage with vendors to gain a better understanding of 
various solutions. Dis